In [ ]:
import time

import pandas as pd
import numpy as np

from nba_api.stats.endpoints import leaguedashplayerstats, commonteamroster, leaguedashplayerbiostats
from nba_api.stats.static import teams

# Get raw data

In [ ]:
def get_player_stats(first_season: int, last_season: int) -> pd.DataFrame:
    assert first_season <= last_season, 'Last season must be later than first season'

    # Initialize a list to append data for each season
    list_seasons_data = []

    for season in range(first_season, last_season + 1):
        print(f"Fetching {season}...")

        # Initialize list to append data for each type of measure
        list_measure_data = []

        for measure_type in ['Base']:#, 'Advanced', 'Scoring']:
            # Call API
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star='Regular Season',
                measure_type_detailed_defense=measure_type
            )

            # Store the data
            df_stats = stats.get_data_frames()[0]

            # Drop _RANK columns
            # since they just represent position by different criteria
            for col in df_stats.columns:
                if "RANK" in col:
                    df_stats.drop(col, axis=1, inplace=True)

            # Drop the MIN column in Advanced and Scoring
            # because they are minutes per game instead of total minutes
            if measure_type != 'Base':
                df_stats.drop('MIN', axis=1, inplace=True)

            # Append data to list of dataframes for the season
            list_measure_data.append(df_stats)

            # Sleep to not overload API calls
            time.sleep(0.5)
        
        # Merge the data for the season
        df_season = list_measure_data[0]
        for df_extra in list_measure_data[1:]:
            df_season = df_season.merge(df_extra)

        # Add SEASON column to identify data when we concatenate all seasons together
        df_season['SEASON'] = season

        # Append data to the list of dataframes for all seasons
        list_seasons_data.append(df_season)

    # Concatenate the data together at set an ID for each player + season combination    
    df = pd.concat(list_seasons_data, ignore_index=True)
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df

In [ ]:
def get_player_positions(first_season: int, last_season: int) -> pd.DataFrame:
    all_teams = teams.get_teams()
    rosters = []

    for team in all_teams:
        for season in range(first_season, last_season + 1):
            roster = commonteamroster.CommonTeamRoster(
                team_id=team['id'],
                season=season
            )
            df_roster = roster.get_data_frames()[0]
            df_roster['SEASON'] = season

            rosters.append(roster.get_data_frames()[0])
            time.sleep(0.5)  # avoid rate limiting

    df = pd.concat(rosters)[['PLAYER_ID', 'SEASON', 'POSITION']]
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df[['ID', 'POSITION']]

In [ ]:
def get_player_bio(first_season: int, last_season: int) -> pd.DataFrame:
    assert first_season <= last_season, 'Last season must be later than first season'

    # Initialize a list to append data for each season
    list_seasons_data = []

    for season in range(first_season, last_season + 1):
        print(f"Fetching {season}...")

        # Call API
        bio = leaguedashplayerbiostats.LeagueDashPlayerBioStats(season=season)

        # Store the data
        df_season = bio.get_data_frames()[0][['PLAYER_ID', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT']]

        # Sleep to not overload API calls
        time.sleep(0.5)

        # Add SEASON column to identify data when we concatenate all seasons together
        df_season['SEASON'] = season

        # Append data to the list of dataframes for all seasons
        list_seasons_data.append(df_season)

    # Concatenate the data together at set an ID for each player + season combination    
    df = pd.concat(list_seasons_data, ignore_index=True)
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    # Rename columns
    df.rename(
        columns={'PLAYER_HEIGHT_INCHES': 'HEIGHT', 'PLAYER_WEIGHT': 'WEIGHT'},
        inplace=True
    )

    return df[['ID', 'HEIGHT', 'WEIGHT']]

In [ ]:
df_stats_raw = pd.read_csv("../assets/nba_player_stats.csv")
# df_stats_raw = get_player_stats(first_season=2016, last_season=2025)

In [ ]:
df_positions_raw = pd.read_csv("../assets/nba_player_positions.csv")
# df_positions_raw = get_player_positions(first_season=2016, last_season=2025)

In [ ]:
df_bio_raw = pd.read_csv("../assets/nba_player_bio.csv")
# df_bio_raw = get_player_bio(first_season=2016, last_season=2025)

In [ ]:
df_raw = df_stats_raw.merge(df_positions_raw, how="left").merge(df_bio_raw, how="left")

# Data cleaning

## Row treatment

Basketball statistics are highly dependent on sample size. When a player only plays 15 total minutes across a whole season, their advanced rates and percentages explode into unrealistic, hyper-volatile extremes.

We choose 250 minutes (equivalent to 10 games with 25 minutes of play)

In [ ]:
df_raw = df_raw[df_raw["MIN"] >= 250]

## Columns treatment

### Drop Unneeded

In [ ]:
df_raw = df_raw.drop(columns=['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'SEASON', 'TEAM_COUNT', 'GP', 'W', 'L', 'W_PCT'])

This leaves only `ID` and `POSITION` as string columns. All the remaining ones are numeric.

### Normalize (per 36 minutes)

Totals suffer from volume bias if we don't normalize them. We follow the standard of giving stats per 36 minutes.

In [ ]:
for col in ['PTS', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV', 'BLKA', 'PF', 'PFD']:
    df_raw[f'{col}_PER36'] = df_raw[col] / df_raw['MIN'] * 36
    df_raw.drop(columns=[col], inplace=True)

df_raw.drop(columns='MIN', inplace=True)

### Correlated

In [ ]:
columns_to_drop = [
    'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',   # Redundant with shooting percentages
    'REB',                                        # Redundant with OREB and DFREB
    'PLUS_MINUS',                                 # Highly dependent on team quality
    'DD2', 'TD3',                                 # Double-dobules and triple-doubles are correlated with other metrics
    'NBA_FANTASY_PTS', 'WNBA_FANTASY_PTS'         # Computed directly from a formula using other stats
]

In [ ]:
df_raw.drop(columns=columns_to_drop, inplace=True)

### Imputation

There are 3 players with no listed weight. We just impute the median weight of players with the same height.

In [ ]:
df_raw['WEIGHT'] = df_raw.groupby('HEIGHT')['WEIGHT'].transform(lambda x: x.fillna(x.median()))

# Exploratory data analysis

In [ ]:
df_raw[df_raw.isna().any(axis=1)]